# Simpsons RAG → small local model (LoRA distillation) on Colab

Runtime → Change runtime type → **T4 GPU**. Run the cells in order.
The logic lives in `src/distill/` (tested offline); this notebook only calls it.

Before starting, put `train.jsonl`, `val.jsonl` and `test.jsonl` (made locally with
`python -m distill.build_dataset`) in a Drive folder, by default `MyDrive/simpsons_distill/data/`.
They are git-ignored, so they are not in the repo.

In [ ]:
# 1) Settings -------------------------------------------------------------
MODEL    = "unsloth/Qwen3-1.7B"      # other candidates: "unsloth/Gemma-3-4b-it", "unsloth/Llama-3.2-1B-Instruct"
REPO_URL = "https://github.com/JimenezPena/SimpsonsTriviaRAG"
DRIVE    = "/content/drive/MyDrive/simpsons_distill"
SLUG     = MODEL.split("/")[-1].lower()
DATA     = f"{DRIVE}/data"
OUT      = f"{DRIVE}/{SLUG}"
SRC      = "/content/repo/src"

In [ ]:
# 2) Drive + code ---------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(OUT, exist_ok=True)
if not os.path.exists("/content/repo"):
    !git clone --depth 1 {REPO_URL} /content/repo
assert os.path.exists(f"{SRC}/distill"), (
    "src/distill is not in the repo yet: apply the patches, push them from your machine, "
    "or copy the distill/ folder from Drive into /content/repo/src/")
for f in ("train.jsonl", "val.jsonl", "test.jsonl"):
    assert os.path.exists(f"{DATA}/{f}"), f"missing {DATA}/{f}"
!wc -l {DATA}/*.jsonl

In [ ]:
# 3) Install (Colab already has torch) ------------------------------------
!pip install -q unsloth sacrebleu

# Qwen3.5 only: it needs transformers v5, which Colab does not ship by default,
# and Unsloth advises 16-bit LoRA instead of QLoRA for it. Uncomment, then
# restart the runtime. (Command taken from Unsloth's Qwen3.5 guide; unverified here.)
# !pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

## 4) Base model, zero-shot (the "base student" row)
Also tells you whether the model understands the task and the language before any training.
Use **val** for choosing models and keep **test** for the final numbers.

In [ ]:
!PYTHONPATH={SRC} python -m distill.eval_student --model {MODEL} \
    --data {DATA}/val.jsonl --limit 30 --out {OUT}/preds_base_val.jsonl

## 5) Smoke test, then train
Five steps first, to catch errors cheaply. If it works, delete the smoke-test checkpoints
(`{OUT}/checkpoints`) before the real run, or it will resume from them.

In [ ]:
!PYTHONPATH={SRC} python -m distill.train_lora --model {MODEL} \
    --train {DATA}/train.jsonl --val {DATA}/val.jsonl --out {OUT}/smoke --max-steps 5

In [ ]:
# Real run: resumable. If the session is cut, reconnect, run cells 1-3 and this one again.
!PYTHONPATH={SRC} python -m distill.train_lora --model {MODEL} \
    --train {DATA}/train.jsonl --val {DATA}/val.jsonl --out {OUT} --epochs 2

## 6) LoRA student, then the comparison table

In [ ]:
!PYTHONPATH={SRC} python -m distill.eval_student --model {MODEL} --adapter {OUT}/adapter \
    --data {DATA}/val.jsonl --limit 30 --out {OUT}/preds_lora_val.jsonl

In [ ]:
!PYTHONPATH={SRC} python -m distill.metrics --data {DATA}/val.jsonl \
    --preds base={OUT}/preds_base_val.jsonl lora={OUT}/preds_lora_val.jsonl

Final numbers: repeat 4 and 6 with `test.jsonl` (no `--limit`) once you have picked the model.
Read some answers by hand (`preds_*.jsonl`) next to the teacher's: the metrics are proxies.